This notebook runs the Watershed algorithm on the Watershed-SNV annotation pipeline.

The amount of time this notebook takes to run depends on the subset of chromosomes you chose when running the annotation workflow. If you chose all chromosomes, it may take about 20minutes to run. Otherwise, less than 5 minutes.

## Setup

In [1]:
library(AnVILGCP)
library(data.table)

# Watershed requirements
install.packages(c('PRROC','optparse','lbfgs'))

Installing packages into ‘/home/jupyter/packages’
(as ‘lib’ is unspecified)

also installing the dependency ‘getopt’




### `git clone` and patch Watershed code
The Watershed code is missing `#include <random>` in one of its C++ files; this is patched in.

In [2]:
system('git clone --depth 1 https://github.com/BennyStrobes/Watershed.git')
system("sed -i '1i #include <random>' Watershed/crf_variational_updates.cpp", intern=T)

character(0)

In [3]:
head(fread('collapsed-reduced.tsv')) # Will be the watershed input

SubjectID,GeneName,minMAF,minTSSdist,minTESdist,n_rarevars_near_gene,GC,CpG,priPhCons,mamPhCons,⋯,intron_variant,missense_variant,synonymous_variant,non_coding_transcript_exon_variant,splice_acceptor_variant,splice_donor_variant,splice_region_variant,stop_gained,p,N2pair
<chr>,<chr>,<dbl>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<int>
NA18879,ENSG00000258410,0.001851850,570,2302,4,0.669,0.053,0.325,0.324,⋯,1,0,0,1,0,0,0,0,-0.04855431,238
NA19064,ENSG00000258410,0.004629630,24168,4960,1,0.563,0.013,0.003,0.003,⋯,0,0,0,0,0,0,0,0,-0.20715218,NA
HG02282,ENSG00000258410,0.000925926,570,489,9,0.689,0.053,0.329,0.200,⋯,1,0,0,1,0,0,0,0,0.99847261,NA
HG02614,ENSG00000258410,0.004629630,23620,4412,2,0.563,0.013,0.003,0.003,⋯,0,0,0,0,0,0,0,0,0.89998252,NA
HG03268,ENSG00000258410,0.004629630,570,3002,3,0.669,0.053,0.329,0.034,⋯,1,0,0,1,0,0,0,0,0.18021189,NA
NA19648,ENSG00000258410,0.001851850,2642,1264,3,0.576,0.027,0.037,0.037,⋯,1,0,0,0,0,0,0,0,0.65288025,166


# Run `predict_watershed.R`

In [4]:
old_wd <- setwd('Watershed')
system(paste('Rscript predict_watershed.R',
  '--training_input    ../collapsed-reduced.tsv',
  '--prediction_input  ../collapsed-reduced.tsv',
  '--number_dimensions 1',
  '--output_prefix     watershed_output',
  '--model_name        Watershed_exact'
),intern=T)
system('gcloud storage cp watershed_output_* $WORKSPACE_BUCKET/data/derived/')
out <- fread('watershed_output_posterior_probability.txt')
setwd(old_wd)

[1] "Only RIVER can be run on data with 1 dimension."
 [2] " Changing model to RIVER."                      
 [3] "########################"                       
 [4] "ITERATION 1"                                    
 [5] "Average total norm: 0.00124976033838206"        
 [6] "########################"                       
 [7] "ITERATION 2"                                    
 [8] "Average total norm: 0.000927220844438395"       
 [9] "########################"                       
[10] "ITERATION 3"                                    
[11] "Average total norm: 0.000722014571799696"       
[12] "########################"                       
[13] "ITERATION 4"                                    
[14] "Average total norm: 0.000584180440659356"       
[15] "########################"                       
[16] "ITERATION 5"                                    
[17] "Average total norm: 0.000485258054486808"       
[18] "########################"                       
[19] "ITERATION 6"                                    
[20] "Average total norm: 0.000404774363485459"       
[21] "########################"                       
[22] "ITERATION 7"                                    
[23] "Average total norm: 0.000338727142883299"       
[24] "########################"                       
[25] "ITERATION 8"                                    
[26] "Average total norm: 0.000284198358156418"       
[27] "########################"                       
[28] "ITERATION 9"                                    
[29] "Average total norm: 0.000234077703442139"       
[30] "########################"                       
[31] "ITERATION 10"                                   
[32] "Average total norm: 0.000199822635365145"       
[33] "########################"                       
[34] "ITERATION 11"                                   
[35] "Average total norm: 0.000172005615642157"       
[36] "########################"                       
[37] "ITERATION 12"                                   
[38] "Average total norm: 0.000149591637778269"       
[39] "########################"                       
[40] "ITERATION 13"                                   
[41] "Average total norm: 0.000131525361004792"       
[42] "########################"                       
[43] "ITERATION 14"                                   
[44] "Average total norm: 0.000121735415562474"       
[45] "########################"                       
[46] "ITERATION 15"                                   
[47] "Average total norm: 0.00010809333624107"        
[48] "########################"                       
[49] "ITERATION 16"                                   
[50] "Average total norm: 9.60818128819015e-05"       
[51] "Watershed converged"

Below are the posterior probabilities output for each gene-individual pair:

In [5]:
head(out)

sample_names,Watershed_posterior_outlier_signal_1
<chr>,<dbl>
NA18879:ENSG00000258410,0.08836099
NA19064:ENSG00000258410,0.08028185
HG02282:ENSG00000258410,0.08146211
HG02614:ENSG00000258410,0.08033795
HG03268:ENSG00000258410,0.08111429
NA19648:ENSG00000258410,0.08045853
